# 14 · Train on verified tool calls

An analyst asks to check the printed total of invoice **SF-429916** against **GBP 2,349.76**.
Their notes mention **SF-862527**. Both invoices exist. Which record should the ledger check,
and can a successful response prove the right one was checked?

Compare two small adapters trained on the same ledger requests. Separate JSON structure,
successful execution, correct lookup, and the evidence in the terminal response.

[Read the lesson](https://docs.recohut.com/courses/advanced-llm-fine-tuning/3-adapt-models/14-train-on-verified-tool-calls)

## Runtime

Select a Tesla T4 in Colab. Each branch runs in sequence. Model downloads use the
public Hugging Face checkpoint, pinned by revision. The shared runtime can reuse
its weight cache. The material below embeds the invoice fixtures and training code.

In [1]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *['transformers==4.51.3', 'peft==0.15.2', 'accelerate==1.6.0']], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers==4.51.3', 'peft==0.15.2', 'accelerate==1.6.0'], returncode=0)

In [2]:
import torch
if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("Select a Tesla T4 runtime in Colab before running this notebook.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

GPU: Tesla T4
PyTorch: 2.14.1+cu130


## Invoice fixtures

The synthetic input and gold record come from a versioned scenario. The scorer compares
generated JSON with the gold record independently of model training. The release set stays
out of this lesson.

In [3]:
"""Versioned synthetic invoices and independent record scoring for the course."""

from __future__ import annotations

import json
import random
import re
from collections import Counter
from dataclasses import dataclass
from datetime import date, timedelta
from typing import Any

CATALOG_VERSION = "course-invoice-v1"
SPLIT_SIZES = {"train": 576, "dev": 72, "familiar": 96, "novel": 96, "independent": 48, "release": 120}
VISIBLE_CONTRACT = (
    "Extract exactly one JSON object with invoice_id, issued_on (YYYY-MM-DD), supplier, "
    "currency (ISO code), line_items (description, quantity, unit_minor), subtotal_minor, "
    "tax_minor, and total_minor. Preserve line order and convert currency amounts to integer cents. "
    "Use null only for a missing or ambiguous field. Do not rewrite printed amounts, invent fields, or add explanations."
)

@dataclass(frozen=True)
class Line:
    description: str
    quantity: int
    unit_minor: int


@dataclass(frozen=True)
class Invoice:
    key: str
    family: str
    supplier: str
    issued_on: str
    currency: str
    items: tuple[Line, ...]
    tax_minor: int | None
    locale: str
    split: str
    layout: str = "register"
    printed_total_minor: int | None = None


def invoice_record(invoice: Invoice) -> dict[str, Any]:
    subtotal = sum(line.quantity * line.unit_minor for line in invoice.items)
    total = subtotal + invoice.tax_minor if invoice.tax_minor is not None else None
    if invoice.printed_total_minor is not None:
        total = invoice.printed_total_minor
    return {
        "invoice_id": invoice.key,
        "issued_on": invoice.issued_on,
        "supplier": invoice.supplier,
        "currency": invoice.currency,
        "line_items": [
            {"description": item.description, "quantity": item.quantity, "unit_minor": item.unit_minor}
            for item in invoice.items
        ],
        "subtotal_minor": subtotal,
        "tax_minor": invoice.tax_minor,
        "total_minor": total,
    }


def make_catalog() -> list[Invoice]:
    rng = random.Random(20261126)
    sources = {
        "train": (24, ("register", "memo", "checkout", "ledger")),
        "dev": (6, ("register", "memo", "checkout", "ledger")),
        "familiar": (8, ("register", "memo", "checkout", "ledger")),
        "novel": (8, ("packing", "remittance", "service", "notice")),
        "independent": (4, ("separate",)),
        "release": (10, ("packing", "remittance", "service", "notice", "separate")),
    }
    locales = (("en-US", "USD"), ("en-GB", "GBP"), ("de-DE", "EUR"), ("fr-FR", "EUR"), ("en-US", "CAD"))
    services = ("Cooling filter", "Cable sleeve", "Bench calibration", "Door seal", "Grounding clip",
                "Rack inspection", "Meter label", "Packing insert", "Carrier adjustment", "Mounting plate")
    catalog = []
    seen_keys = set()
    supplier_codes = rng.sample(range(1000, 9999), sum(n for n, _ in sources.values()))
    supplier_index = 0
    for split, (family_count, layouts) in sources.items():
        for family_number in range(family_count):
            supplier_code = supplier_codes[supplier_index]
            supplier_index += 1
            layout = layouts[family_number % len(layouts)]
            family = f"{split}-{family_number:02d}"
            locale, currency = locales[family_number % len(locales)]
            for row in range(SPLIT_SIZES[split] // family_count):
                code = f"SF-{rng.randrange(100000, 1000000)}"
                while code in seen_keys:
                    code = f"SF-{rng.randrange(100000, 1000000)}"
                seen_keys.add(code)
                items = tuple(
                    Line(kind, rng.randint(1, 9), rng.randrange(200, 24800))
                    for kind in rng.sample(services, rng.randrange(1, 4))
                )
                subtotal = sum(x.quantity * x.unit_minor for x in items)
                tax = (subtotal * (7 if currency == "USD" else 13) + 50) // 100
                if split in {"novel", "independent", "release"} and row % 12 == 4:
                    tax = None
                displayed = subtotal + tax + 101 if tax is not None and row % 12 == 7 else None
                issued_on = (date(2025, 1, 1) + timedelta(days=rng.randint(0, 590))).isoformat()
                catalog.append(Invoice(code, family, f"Sample Supplier {supplier_code}", issued_on,
                                       currency, items, tax, locale, split, layout, displayed))
    return catalog


def formatted_money(minor: int | None, locale: str) -> str:
    if minor is None:
        return "-"
    dollars, cents = divmod(minor, 100)
    number = f"{dollars:,}.{cents:02d}"
    if locale == "de-DE":
        return number.replace(",", "_").replace(".", ",").replace("_", ".")
    if locale == "fr-FR":
        return number.replace(",", " ").replace(".", ",")
    return number


def invoice_text(invoice: Invoice) -> str:
    gold = invoice_record(invoice)
    money = lambda value: formatted_money(value, invoice.locale)
    identifier = f"{invoice.key}"
    supplier = invoice.supplier
    issued = invoice.issued_on
    currency = invoice.currency
    subtotal = money(gold["subtotal_minor"])
    tax = money(gold["tax_minor"])
    total = money(gold["total_minor"])
    register = [f"{row.description} | {row.quantity} | {money(row.unit_minor)}" for row in invoice.items]
    inline = [f"{row.quantity} x {row.description} @ {money(row.unit_minor)} per unit" for row in invoice.items]
    service = [f"{row.description}: {money(row.unit_minor)} each for {row.quantity} units" for row in invoice.items]
    if invoice.layout == "register":
        return "\n".join([f"Supplier: {supplier}", f"Invoice: {identifier}", f"Issued: {issued}", f"Currency: {currency}",
                          "Description | Quantity | Unit price", *register,
                          f"Subtotal: {subtotal}", f"Tax: {tax}", f"Amount due: {total}"])
    if invoice.layout == "memo":
        return "\n".join([f"{supplier} billing memo / {issued}", f"Account invoice {identifier} [{currency}]",
                          *inline, f"Total due {total} including tax {tax}; net amount {subtotal}."])
    if invoice.layout == "checkout":
        return "\n".join([f"{identifier} / supplier {supplier}", f"Date of issue {issued} / All amounts {currency}",
                          *service, f"Net {subtotal}", f"Sales tax {tax}", f"Payable {total}"])
    if invoice.layout == "ledger":
        return "\n".join([f"{supplier} | issued {issued} | invoice {identifier}", f"Ledger in {currency}",
                          "Service | Count | Per unit", *register, f"Before tax {subtotal}",
                          f"Tax charged {tax}", f"Balance {total}"])
    if invoice.layout == "packing":
        return "\n".join([f"Billing details / {identifier}", f"Sold by {supplier}", f"Document date {issued}",
                          f"Invoice currency {currency}", f"Balance payable {total}", *service,
                          f"Merchandise {subtotal}", f"Tax {tax}"])
    if invoice.layout == "remittance":
        return "\n".join([f"Remittance advice for {identifier} ({currency})", f"{supplier} / Document issued {issued}",
                          f"Pay this amount {total}", *inline, f"Net charges {subtotal} / Tax {tax}"])
    if invoice.layout == "service":
        return "\n".join([f"Ref {identifier} / {currency}", f"Service fee notice, {issued}", supplier,
                          *[f"Unit {money(row.unit_minor)} / {row.quantity} times / {row.description}" for row in invoice.items],
                          f"Fee subtotal {subtotal}", f"Plus tax {tax}", f"Amount requested {total}"])
    if invoice.layout == "notice":
        return "\n".join([f"{supplier} / statement {identifier}", f"Issued on {issued}",
                          f"Invoice denomination {currency}", "One-off charges", *service,
                          f"Pre-tax {subtotal} / Tax assessment {tax} / Final bill {total}"])
    return "\n".join([f"Document {identifier}", f"Biller {supplier} / As of {issued} / {currency}", *inline,
                      f"Gross {total}", f"Breakdown: net {subtotal} and taxes {tax}."])


def _strict_dict(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    result: dict[str, Any] = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key {key!r}")
        result[key] = value
    return result


def parse_record(completion: str | dict[str, Any]) -> dict[str, Any] | None:
    if isinstance(completion, str):
        try:
            data = json.loads(completion, object_pairs_hook=_strict_dict)
        except (ValueError, TypeError, RecursionError):
            return None
    else:
        data = completion
    keys = {"invoice_id", "issued_on", "supplier", "currency", "line_items", "subtotal_minor", "tax_minor", "total_minor"}
    if not isinstance(data, dict) or set(data) != keys:
        return None
    for name in ("invoice_id", "supplier", "issued_on", "currency"):
        value = data[name]
        if value is not None and (not isinstance(value, str) or not value.strip()):
            return None
    try:
        if data["issued_on"] is not None and date.fromisoformat(data["issued_on"]).isoformat() != data["issued_on"]:
            return None
    except ValueError:
        return None
    if data["currency"] is not None and re.fullmatch("[A-Z]{3}", data["currency"]) is None:
        return None
    for name in ("subtotal_minor", "tax_minor", "total_minor"):
        if data[name] is not None and type(data[name]) is not int:
            return None
    if not isinstance(data["line_items"], list):
        return None
    for row in data["line_items"]:
        if not isinstance(row, dict) or set(row) != {"description", "quantity", "unit_minor"}:
            return None
        if type(row["description"]) is not str or type(row["quantity"]) is not int or type(row["unit_minor"]) is not int:
            return None
        if not row["description"].strip() or row["quantity"] <= 0:
            return None
    return data


def score_record(completion: str | dict[str, Any], invoice: Invoice) -> dict[str, Any]:
    truth = invoice_record(invoice)
    observed = parse_record(completion)
    field_correct = {key: observed is not None and observed.get(key) == value
                     for key, value in truth.items() if key != "line_items"}
    keys = ("description", "quantity", "unit_minor")
    expected = Counter(tuple(r[k] for k in keys) for r in truth["line_items"])
    predicted = Counter(tuple(r[k] for k in keys) for r in observed["line_items"]) if observed is not None else Counter()
    matches = (expected & predicted).total()
    row_count = expected.total() + predicted.total()
    return {"valid": observed is not None, "exact": observed == truth,
            "fields": sum(field_correct.values()) / len(field_correct), "field_correct": field_correct,
            "row_f1": 2 * matches / row_count if row_count else 1.0}


def rules_extract(text: str) -> dict[str, Any] | None:
    lines = text.splitlines()
    if len(lines) < 9 or lines[4] != "Description | Quantity | Unit price":
        return None
    property_names = ("supplier", "invoice_id", "issued_on", "currency")
    headers = ("Supplier: ", "Invoice: ", "Issued: ", "Currency: ")
    if any(not line.startswith(header) for line, header in zip(lines, headers)):
        return None
    data = {name: line.removeprefix(header) for name, line, header in zip(property_names, lines, headers)}

    def minor(text_amount: str) -> int | None:
        if text_amount == "-":
            return None
        if re.fullmatch(r"(?:\d+|\d{1,3}(?:[., ]\d{3})+)[.,]\d{2}", text_amount) is None:
            raise ValueError("Unsupported amount format")
        units, frac = text_amount[:-3], text_amount[-2:]
        return int(re.sub(r"\D", "", units)) * 100 + int(frac)

    try:
        rows = [line.split(" | ") for line in lines[5:-3]]
        data["line_items"] = [{"description": desc, "quantity": int(qty), "unit_minor": minor(unit)}
                              for desc, qty, unit in rows]
        data["subtotal_minor"] = minor(lines[-3].removeprefix("Subtotal: "))
        data["tax_minor"] = minor(lines[-2].removeprefix("Tax: "))
        data["total_minor"] = minor(lines[-1].removeprefix("Amount due: "))
    except ValueError:
        return None
    return parse_record(data)


def examples_for(split: str, limit: int | None = None) -> list[Invoice]:
    found = [invoice for invoice in make_catalog() if invoice.split == split]
    if not found:
        raise ValueError(f"Unknown invoice split {split!r}")
    if limit is None or limit >= len(found):
        return found
    if limit <= 0:
        raise ValueError("Limit must be positive")
    groups: dict[str, list[Invoice]] = {}
    for invoice in found:
        groups.setdefault(invoice.family, []).append(invoice)
    balanced = [invoice for group_rows in zip(*groups.values(), strict=True) for invoice in group_rows]
    return balanced[:limit]

## Completion masking

The prefix-stable masking and preference helpers below adapt Apache-2.0 recipe code from
the Together Post-Training Cookbook. The shared helpers build a causal training sequence
while marking just the answer tokens and end marker as targets.

In [4]:
# SPDX-License-Identifier: Apache-2.0
"""Portable training functions from the Together Post-Training Cookbook.

Adapted from togethercomputer/post-training-cookbook at
226e03ca225d6f13434047cf0f45ab3070a29801. See third_party for the Apache licence
and the original paths. The hosted training and provisioning code is replaced
by notebook-local PyTorch and typed fixtures.
"""

from __future__ import annotations

import random
from collections import Counter
from collections.abc import Callable, Iterable, Sequence
from dataclasses import dataclass, field
from typing import NamedTuple, TypeVar

import torch
import torch.nn.functional as F


class Turn(NamedTuple):
    prompt: list[int]
    completion: list[int]


class MaskedSequence(NamedTuple):
    input_ids: list[int]
    completion_mask: list[int]


def split_turns(turns: Iterable[Turn]) -> list[MaskedSequence]:
    """Build prefix-stable sequences with a token-aligned completion mask."""
    sequences: list[MaskedSequence] = []
    emitted: list[int] = []
    mask: list[int] = []
    for prompt, completion in turns:
        if len(prompt) < len(emitted) or prompt[: len(emitted)] != emitted:
            if emitted:
                sequences.append(MaskedSequence(emitted, mask))
            mask = [0] * len(prompt)
        else:
            mask = mask + [0] * (len(prompt) - len(emitted))
        emitted = prompt + completion
        mask = mask + [1] * len(completion)
    if emitted:
        sequences.append(MaskedSequence(emitted, mask))
    return sequences


def masked_logp_sum(logprobs: torch.Tensor, mask: Sequence[float]) -> torch.Tensor:
    """Sum target-aligned response log probabilities and retain gradients."""
    if logprobs.ndim != 1 or len(mask) != logprobs.shape[0]:
        raise ValueError("The response mask must match the one-dimensional target log probabilities.")
    weights = torch.as_tensor(mask, dtype=logprobs.dtype, device=logprobs.device)
    return (logprobs * weights).sum()


def make_dpo_loss(
    ref_sums: Sequence[float],
    masks: Sequence[Sequence[float]],
    *,
    beta: float,
    label_smoothing: float = 0.0,
) -> Callable[[Sequence[torch.Tensor]], torch.Tensor]:
    """Close over fixed reference scores and response masks for chosen/rejected pairs."""
    if not ref_sums or len(ref_sums) != len(masks) or len(ref_sums) % 2:
        raise ValueError("Supply at least one complete pair and one mask for each reference score.")
    if beta <= 0.0 or not 0.0 <= label_smoothing <= 0.5:
        raise ValueError("Beta must be positive and label smoothing must be between 0 and 0.5.")
    branch_size = len(ref_sums)

    def loss_fn(policy_logprobs: Sequence[torch.Tensor]) -> torch.Tensor:
        if len(policy_logprobs) != branch_size:
            raise ValueError(f"Expected {branch_size} policy log-probability tensors.")
        policy = torch.stack([masked_logp_sum(values, mask) for values, mask in zip(policy_logprobs, masks, strict=True)])
        reference = policy.new_tensor(ref_sums)
        delta = policy - reference
        margin = beta * (delta[0::2] - delta[1::2])
        return (-(1.0 - label_smoothing) * F.logsigmoid(margin) - label_smoothing * F.logsigmoid(-margin)).mean()

    return loss_fn


def compute_overlong_penalty(
    response_length: int,
    max_response_length: int,
    overlong_buffer_len: int,
    penalty_factor: float = 1.0,
) -> float:
    """Apply the cookbook's DAPO soft penalty near a generation cap."""
    if overlong_buffer_len <= 0:
        return 0.0
    expected_len = max_response_length - overlong_buffer_len
    exceed_len = response_length - expected_len
    return min(-exceed_len / overlong_buffer_len * penalty_factor, 0.0)


Item = TypeVar("Item")


def oversample_until_kept(
    draw_batch: Callable[[], Iterable[Item]],
    *,
    is_kept: Callable[[Item], bool],
    target: int,
    max_attempts: int,
) -> tuple[list[Item], int]:
    """Keep rejected groups in the audit trail while bounding draws."""
    if target <= 0 or max_attempts <= 0:
        raise ValueError("Target and maximum attempts must be positive.")
    items: list[Item] = []
    kept = 0
    for attempts in range(1, max_attempts + 1):
        for item in draw_batch():
            items.append(item)
            if is_kept(item):
                kept += 1
                if kept >= target:
                    return items, attempts
    return items, max_attempts


@dataclass(frozen=True)
class TeacherRequest:
    key: str
    messages: Sequence[dict[str, str]]
    meta: dict[str, object] = field(default_factory=dict)


@dataclass(frozen=True)
class TeacherOutput:
    content: str
    ok: bool = True
    truncated: bool = False


def sft_rows(
    requests: Sequence[TeacherRequest],
    outputs: dict[str, TeacherOutput],
    *,
    reject_reason: Callable[[TeacherRequest, TeacherOutput], str | None] | None = None,
) -> tuple[list[dict[str, object]], Counter[str]]:
    """Convert verified procedural teacher outputs to SFT rows and count exclusions."""
    rows: list[dict[str, object]] = []
    dropped: Counter[str] = Counter()
    for request in requests:
        output = outputs.get(request.key)
        if output is None:
            dropped["missing"] += 1
            continue
        if not output.ok:
            dropped["failed"] += 1
            continue
        if output.truncated:
            dropped["truncated"] += 1
            continue
        content = output.content.strip()
        if not content:
            dropped["empty"] += 1
            continue
        reason = reject_reason(request, output) if reject_reason is not None else None
        if reason is not None:
            dropped[reason] += 1
            continue
        assistant = {"role": "assistant", "content": content}
        rows.append({"id": request.key, "messages": [*request.messages, assistant], "meta": dict(request.meta)})
    return rows, dropped


def mix(*groups: Sequence[Item], seed: int = 0) -> list[Item]:
    """Concatenate row groups and shuffle them deterministically."""
    rows = [row for group in groups for row in group]
    random.Random(seed).shuffle(rows)
    return rows

## Training and scoring

A common prompt, batch schedule and scorer make each comparison interpretable. The paired
uncertainty calculation resamples whole layout families, so repeated layouts do not count
as independent evidence.

In [5]:
"""Shared notebook-local text training and measurement for lessons 01–03."""
from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import json
import random
import statistics
import time
from collections import defaultdict
from pathlib import Path

import torch

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
SEED = 42
MAX_TOKENS = 1536
MAX_NEW_TOKENS = 384
BATCH = 2



def canonical_json(record):
    return json.dumps(record, ensure_ascii=False, separators=(",", ":"))


def ordered_by_family(items):
    by_family = defaultdict(list)
    for invoice in items:
        by_family[invoice.family].append(invoice)
    randomizer = random.Random(SEED)
    families = list(by_family)
    randomizer.shuffle(families)
    for rows in by_family.values():
        randomizer.shuffle(rows)
    return [
        by_family[family][offset]
        for offset in range(max(map(len, by_family.values()), default=0))
        for family in families
        if offset < len(by_family[family])
    ]


def lesson_prompts(catalog):
    """Choose one training exemplar and preserve dataset order after seeded sampling."""
    train = ordered_by_family([invoice for invoice in catalog if invoice.split == "train"])
    if len(train) < 3:
        raise ValueError("The shared catalogue must contain at least three training invoices.")
    exemplar, *train = train
    teach_splits = {"familiar", "novel", "independent"}
    evaluation = [invoice for invoice in catalog if invoice.split in teach_splits]
    if not evaluation or set(item.split for item in evaluation) != teach_splits:
        raise ValueError("The three fixed teaching slices must be present.")
    return exemplar, train, evaluation


def prompt_messages(invoice, exemplar):
    return [
        {"role": "system", "content": VISIBLE_CONTRACT},
        {"role": "user", "content": invoice_text(exemplar)},
        {"role": "assistant", "content": canonical_json(invoice_record(exemplar))},
        {"role": "user", "content": invoice_text(invoice)},
    ]


def encode_prompt(tokenizer, invoice, exemplar):
    return tokenizer.apply_chat_template(
        prompt_messages(invoice, exemplar), tokenize=True, add_generation_prompt=True,
    )


def encode_supervision(tokenizer, invoice, answer, exemplar):
    prompt = encode_prompt(tokenizer, invoice, exemplar)
    completion = tokenizer.encode(answer, add_special_tokens=False) + [tokenizer.eos_token_id]
    sequence, = split_turns([Turn(prompt, completion)])
    if len(sequence.input_ids) > MAX_TOKENS or len(completion) + 1 > MAX_NEW_TOKENS:
        raise ValueError(f"Invoice {invoice.key} exceeds the fixed token budget.")
    return sequence


def right_pad(examples, pad_id, device):
    width = max(len(example.input_ids) for example in examples)
    ids, attention, labels = [], [], []
    for example in examples:
        n_pad = width - len(example.input_ids)
        ids.append(example.input_ids + [pad_id] * n_pad)
        attention.append([1] * len(example.input_ids) + [0] * n_pad)
        labels.append([token if mask else -100 for token, mask in zip(
            example.input_ids, example.completion_mask, strict=True,
        )] + [-100] * n_pad)
    return {
        key: torch.tensor(rows, dtype=torch.long, device=device)
        for key, rows in (("input_ids", ids), ("attention_mask", attention), ("labels", labels))
    }


def load_model(encoding, *, adapter=None, trainable=False):
    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig

    torch.manual_seed(SEED)
    if encoding == "nf4":
        quant = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL, revision=REVISION, quantization_config=quant, torch_dtype=torch.float16,
            device_map={"": 0}, attn_implementation="sdpa",
        )
    elif encoding == "fp16":
        model = AutoModelForCausalLM.from_pretrained(
            MODEL, revision=REVISION, torch_dtype=torch.float16, attn_implementation="sdpa",
        ).to("cuda")
    else:
        raise ValueError(f"Unsupported weight encoding {encoding!r}.")
    if not trainable and adapter is None:
        return model
    if trainable and encoding == "nf4":
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
    if adapter is not None:
        return PeftModel.from_pretrained(model, str(adapter), is_trainable=trainable)
    if not trainable:
        raise ValueError("An adapter is required for evaluating a trained arm.")
    return get_peft_model(model, LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.0, bias="none",
        task_type="CAUSAL_LM", target_modules=["q_proj", "v_proj"],
    ))


def adapter_train(model, schedule, objective, *, learning_rate, use_cuda):
    """Both arms share the same schedule, optimiser and gradient policy."""
    params = [param for param in model.parameters() if param.requires_grad]
    if not params or not schedule:
        raise ValueError("Expected trainable parameters and a nonempty update schedule.")
    optimiser = torch.optim.AdamW(params, lr=learning_rate)
    scaler = torch.amp.GradScaler("cuda", enabled=use_cuda, init_scale=512, growth_interval=10_000)
    torch.manual_seed(SEED)
    model.train()
    model.config.use_cache = False
    if use_cuda:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    losses, supervised_count, batch_token_count, skipped = [], 0, 0, 0
    for batch in schedule:
        optimiser.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_cuda):
            loss, supervised, batch_tokens = objective(model, batch)
        if not torch.isfinite(loss).item():
            raise FloatingPointError("A non-finite training loss invalidates the comparison.")
        scaler.scale(loss).backward()
        scaler.unscale_(optimiser)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        previous_scale = scaler.get_scale()
        scaler.step(optimiser)
        scaler.update()
        skipped += int(scaler.get_scale() < previous_scale)
        losses.append(float(loss.detach().cpu()))
        supervised_count += supervised
        batch_token_count += batch_tokens
    if use_cuda:
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    return {
        "attempted_updates": len(schedule),
        "updates": len(schedule) - skipped,
        "supervised_response_tokens": supervised_count,
        "unpadded_input_tokens": batch_token_count,
        "seconds": elapsed,
        "supervised_tokens_per_second": supervised_count / max(elapsed, 1e-9),
        "peak_gib": torch.cuda.max_memory_allocated() / 2**30 if use_cuda else None,
        "losses": losses,
        "trainable_parameters": sum(parameter.numel() for parameter in params),
    }


def sft_objective(pad_id, device):
    def loss_for(model, examples):
        batch = right_pad(examples, pad_id, device)
        loss = model(**batch).loss
        supervised = int(batch["labels"][:, 1:].ne(-100).sum().item())
        if not supervised:
            raise ValueError("An update needs at least one response token.")
        unpadded = int(batch["attention_mask"].sum().item())
        return loss, supervised, unpadded
    return loss_for


def make_schedule(examples, batch_size=BATCH):
    if not examples or batch_size <= 0:
        raise ValueError("Expected training examples and a positive batch size.")
    return [examples[start : start + batch_size] for start in range(0, len(examples), batch_size)]


def predictions(model, tokenizer, eval_items, exemplar, batch_size=16):
    model.eval()
    device = model.get_input_embeddings().weight.device
    outputs = []
    for start in range(0, len(eval_items), batch_size):
        group = eval_items[start : start + batch_size]
        prompts = [encode_prompt(tokenizer, item, exemplar) for item in group]
        width = max(map(len, prompts))
        if width + MAX_NEW_TOKENS > MAX_TOKENS:
            raise ValueError("Evaluation prompt and fixed decode cap exceed the token budget.")
        ids = torch.tensor([[tokenizer.pad_token_id] * (width - len(p)) + p for p in prompts], device=device)
        mask = torch.tensor([[0] * (width - len(p)) + [1] * len(p) for p in prompts], device=device)
        with torch.inference_mode():
            generated = model.generate(
                input_ids=ids, attention_mask=mask,
                do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                use_cache=True, temperature=None, top_p=None, top_k=None,
            )
        for item, tokens in zip(group, generated[:, width:].tolist(), strict=True):
            stopped = tokenizer.eos_token_id in tokens
            if stopped:
                tokens = tokens[:tokens.index(tokenizer.eos_token_id)]
            answer = tokenizer.decode(tokens, skip_special_tokens=True)
            outputs.append({
                "key": item.key, "split": item.split, "family": item.family,
                "completed": stopped, "answer": answer, "reference": invoice_record(item),
                "completion_tokens": len(tokens), **score_record(answer, item),
            })
    return outputs


def rules_predictions(items):
    rows = []
    for item in items:
        result = rules_extract(invoice_text(item))
        rows.append({
            "key": item.key, "split": item.split, "family": item.family,
            "completed": result is not None, "answer": result, "reference": invoice_record(item),
            **score_record(result if result is not None else "", item),
        })
    return rows


def paired_family_interval(reference, candidate, repeats=1000):
    """Pair by invoice, then resample whole layout families within a slice."""
    deltas = defaultdict(list)
    old_by_key = {row["key"]: row for row in reference}
    if set(old_by_key) != {row["key"] for row in candidate}:
        raise ValueError("Paired comparisons require the same unique invoice keys.")
    for row in candidate:
        old = old_by_key[row["key"]]
        if (old["family"], old["split"]) != (row["family"], row["split"]):
            raise ValueError("An invoice moved family or split.")
        deltas[row["family"]].append(int(row["exact"]) - int(old["exact"]))
    groups = list(deltas.values())
    if len(groups) < 2:
        return None
    randomizer = random.Random(SEED)
    draws = []
    for _ in range(repeats):
        sampled_groups = randomizer.choices(groups, k=len(groups))
        draws.append(sum(map(sum, sampled_groups)) / sum(map(len, sampled_groups)))
    draws.sort()
    return [draws[int(0.025 * (repeats - 1))], draws[int(0.975 * (repeats - 1))]]


def free_gpu():
    """Collect dropped references between sequential Colab arms."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_results(lesson, scores, training, comparisons, **diagnostics):
    import matplotlib
    import matplotlib.pyplot as plt

    seed = SEED
    grouped = {
        name: {
            split: {
                "n": len(rows_in_split),
                **{
                    metric: statistics.mean(float(row[metric]) for row in rows_in_split)
                    for metric in ("valid", "exact", "completed", "fields", "row_f1")
                },
            }
            for split in ("familiar", "novel", "independent")
            if (rows_in_split := [row for row in rows if row["split"] == split])
        }
        for name, rows in scores.items()
    }
    intervals = {}
    for baseline, candidate in comparisons:
        intervals[f"{candidate} minus {baseline}"] = {
            split: interval
            for split in ("familiar", "novel", "independent")
            if (interval := paired_family_interval(
                [row for row in scores[baseline] if row["split"] == split],
                [row for row in scores[candidate] if row["split"] == split],
            )) is not None
        }
    roster = {
        row["key"]: {"split": row["split"], "family": row["family"]}
        for row in next(iter(scores.values()))
    }
    report = {
        "model_id": MODEL, "model_revision": REVISION, "seed": seed,
        "versions": {name: importlib.metadata.version(name) for name in ("torch", "transformers", "peft")},
        "hardware": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU smoke",
        "held_out_roster_sha256": hashlib.sha256(
            json.dumps(roster, sort_keys=True, separators=(",", ":")).encode()
        ).hexdigest(),
        "scores": grouped, "paired_family_intervals": intervals, "training": training,
        "diagnostics": diagnostics,
    }
    payload = Path("results")
    payload.mkdir(exist_ok=True)
    (payload / f"{lesson}-report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    (payload / f"{lesson}-predictions.json").write_text(json.dumps(scores, indent=2) + "\n", encoding="utf-8")
    print(json.dumps({key: value for key, value in report.items() if key != "training"}, indent=2))
    print(json.dumps({
        arm: {key: value for key, value in metrics.items() if key != "losses"}
        for arm, metrics in training.items()
    }, indent=2))

    matplotlib.rcParams["svg.fonttype"] = "none"
    fig, axes = plt.subplots(1, len(grouped[next(iter(grouped))]), figsize=(11, 4), squeeze=False, sharey=True)
    for axis, split in zip(axes[0], grouped[next(iter(grouped))], strict=True):
        names = list(grouped)
        values = [grouped[name][split]["exact"] for name in names]
        axis.barh(names, values, color="#357e72")
        axis.set_title(split.title())
        axis.set_xlim(0, 1)
        axis.set_xlabel("Exact record rate")
        axis.grid(axis="x", alpha=0.2)
    axes[0, 0].invert_yaxis()
    fig.suptitle("Complete-record accuracy on the held-out invoice slices")
    fig.tight_layout()
    fig.savefig(payload / f"{lesson}-exact.svg", bbox_inches="tight")
    from IPython.display import SVG, display
    display(SVG(filename=str(payload / f"{lesson}-exact.svg")))
    plt.close(fig)
    return report

In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
if tokenizer.pad_token_id is None or tokenizer.eos_token_id is None:
    raise ValueError("The pinned tokenizer must supply padding and completion markers.")
catalogue = make_catalog()
exemplar, training_items, evaluation_items = lesson_prompts(catalogue)
print("Teaching evaluation counts:", dict(Counter(x.split for x in evaluation_items)))
print("One train-only prompt exemplar:", exemplar.key)

Teaching evaluation counts: {'familiar': 96, 'novel': 96, 'independent': 48}
One train-only prompt exemplar: SF-898991


## A read-only ledger with a decoy

Each case contains an immutable target and decoy. Lookup is required before a match. Tool
results contain observed values or errors and feed back into the next assistant turn. At
most three assistant decisions fit in an episode. Ambiguous record checks require
clarification, and conceptual questions score as successful only without a ledger call.
The tool runtime and semantic verifier each have independent outcomes.

In [7]:
"""Typed offline invoice tools and a bounded multi-turn execution harness."""
from __future__ import annotations

import hashlib
import re
from dataclasses import asdict, dataclass
from types import MappingProxyType

TOOL_CONTRACTS = {
    "lookup_invoice": ({"invoice_id": str}, "Read one invoice before comparing a value."),
    "match_amount": (
        {"invoice_id": str, "expected_minor": int, "currency": str},
        "Compare the printed total to the amount and currency in the request.",
    ),
    "match_purchase_order": (
        {"invoice_id": str, "purchase_order": str},
        "Compare the ledger purchase-order reference with the request.",
    ),
    "match_tax": (
        {"invoice_id": str, "expected_tax_minor": int},
        "Compare the printed tax to the tax amount in the request.",
    ),
    "match_currency": (
        {"invoice_id": str, "expected_currency": str},
        "Compare the invoice currency code to the currency code in the request.",
    ),
}
TRAIN_TOOLS = ("lookup_invoice", "match_amount", "match_purchase_order")
HELD_TOOLS = ("match_tax", "match_currency")
# Two calls and a final decision fit the full three-turn budget.
TURN_LIMIT = 3


@dataclass(frozen=True)
class Entry:
    invoice_id: str
    currency: str
    total_minor: int | None
    tax_minor: int | None
    purchase_order: str


@dataclass(frozen=True)
class Case:
    key: str
    family: str
    split: str
    kind: str
    request: str
    target: Entry
    decoy: Entry
    requested_value: int | str | None = None


@dataclass(frozen=True)
class Episode:
    actions: tuple
    observations: tuple
    final: dict | None
    malformed: bool = False


def to_entry(invoice):
    gold = invoice_record(invoice)
    # Use a namespace disjoint from invoice and vendor identifiers.
    suffix = hashlib.sha256(("purchase-order:" + invoice.key).encode()).hexdigest()[:9].upper()
    return Entry(invoice.key, invoice.currency, gold["total_minor"], gold["tax_minor"], f"PO-{suffix}")


def case_for(invoice, decoy, kind, *, mismatch=False):
    target, distracting = to_entry(invoice), to_entry(decoy)
    misleading_context = f"The notes also mention {distracting.invoice_id}. That is a different invoice."
    requests = {
        "lookup": f"Check whether the ledger contains {target.invoice_id}.",
        "clarification": "Check the invoice amount against 1240 minor units in USD. The invoice identifier is missing.",
        "no call": "Explain in general what an invoice identifier is. There is no record to check.",
    }
    requested = None
    if kind == "amount":
        requested = target.total_minor + (101 if mismatch else 0) if target.total_minor is not None else 1240
        request = (
            f"Check that {target.invoice_id} has a printed total of {requested} minor units in "
            f"{target.currency}. {misleading_context}"
        )
    elif kind == "purchase order":
        requested = distracting.purchase_order if mismatch else target.purchase_order
        request = f"Verify whether {target.invoice_id} belongs to {requested}. {misleading_context}"
    elif kind == "tax":
        requested = target.tax_minor + (10 if mismatch else 0) if target.tax_minor is not None else 120
        request = f"Check the printed tax on {target.invoice_id} against {requested} minor units. {misleading_context}"
    elif kind == "currency":
        requested = {"USD": "EUR", "EUR": "GBP", "GBP": "USD"}.get(target.currency, "USD") if mismatch else target.currency
        request = f"Check whether {target.invoice_id} is denominated in {requested}. {misleading_context}"
    else:
        request = requests[kind]
        if kind == "lookup":
            request += " " + misleading_context
    return Case(invoice.key, invoice.family, invoice.split, kind, request, target, distracting, requested)


def cases_for(items, *, unseen_tool=False):
    kinds = ["lookup", "amount", "purchase order", "clarification", "no call"]
    if unseen_tool:
        kinds += ["tax", "currency", "tax", "currency"]
    next_by_split = defaultdict(list)
    for invoice in items:
        next_by_split[invoice.split].append(invoice)
    cases = []
    for group in next_by_split.values():
        if len(group) < 2:
            raise ValueError("A split needs distinct target and decoy invoices.")
        for index, invoice in enumerate(group):
            kind = kinds[index % len(kinds)]
            cases.append(case_for(
                invoice, group[(index + 1) % len(group)], kind,
                mismatch=(index // len(kinds)) % 2 == 1,
            ))
    return cases


def wire_call(tool, **args):
    return {"call": tool, "args": args}


def wire_final(decision, invoice_id=None, value=None):
    return {"final": decision, "invoice_id": invoice_id, "value": value}


def parse_wire(payload):
    try:
        action = json.loads(payload, object_pairs_hook=_strict_dict) if isinstance(payload, str) else payload
    except (TypeError, ValueError, RecursionError):
        return None
    if not isinstance(action, dict):
        return None
    fields = set(action)
    if fields == {"call", "args"} and type(action["call"]) is str and isinstance(action["args"], dict):
        return action
    if fields != {"final", "invoice_id", "value"}:
        return None
    if (
        not isinstance(action["final"], str)
        or action["final"] not in {"found", "match", "mismatch", "clarify", "no_call"}
        or type(action["invoice_id"]) not in {str, type(None)}
        or type(action["value"]) not in {int, str, type(None)}
    ):
        return None
    return action


class Ledger:
    """The backing records are immutable and the runtime scope is case-local."""

    def __init__(self, case, allowed):
        self.entries = MappingProxyType({entry.invoice_id: entry for entry in (case.target, case.decoy)})
        self.allowed = frozenset(allowed)
        self.looked_up = set()

    def execute(self, action):
        name, args = action["call"], action["args"]
        if name not in self.allowed:
            return {"ok": False, "error": "unknown_tool"}
        schema, _ = TOOL_CONTRACTS[name]
        if set(args) != set(schema) or any(type(args[key]) is not t for key, t in schema.items()):
            return {"ok": False, "error": "invalid_arguments"}
        entry = self.entries.get(args["invoice_id"])
        if entry is None:
            return {"ok": False, "error": "unknown_invoice"}
        if name == "lookup_invoice":
            self.looked_up.add(entry.invoice_id)
            return {"ok": True, "invoice_id": entry.invoice_id, "retrieved": True}
        if entry.invoice_id not in self.looked_up:
            return {"ok": False, "error": "lookup_required"}
        if name == "match_amount":
            observed = entry.total_minor
            matches = observed is not None and observed == args["expected_minor"] and entry.currency == args["currency"]
        elif name == "match_purchase_order":
            observed = entry.purchase_order
            matches = observed == args["purchase_order"]
        elif name == "match_tax":
            observed = entry.tax_minor
            matches = observed is not None and observed == args["expected_tax_minor"]
        else:
            observed = entry.currency
            matches = observed == args["expected_currency"]
        return {"ok": True, "invoice_id": entry.invoice_id, "match": matches, "value": observed}


def execute_episode(case, allowed, choose):
    """Execute the model's decision, feed the actual observation back, and stop on a final."""
    ledger = Ledger(case, allowed)
    actions, observations = [], []
    for _ in range(TURN_LIMIT):
        decision = parse_wire(choose(tuple(actions), tuple(observations)))
        if decision is None:
            return Episode(tuple(actions), tuple(observations), None, malformed=True)
        if "final" in decision:
            return Episode(tuple(actions), tuple(observations), decision)
        actions.append(decision)
        observations.append(ledger.execute(decision))
    return Episode(tuple(actions), tuple(observations), None)


def wanted_calls(case):
    invoice_id = case.target.invoice_id
    lookup = wire_call("lookup_invoice", invoice_id=invoice_id)
    if case.kind == "lookup":
        return [lookup]
    if case.kind == "amount":
        return [lookup, wire_call(
            "match_amount", invoice_id=invoice_id,
            expected_minor=case.requested_value, currency=case.target.currency,
        )]
    if case.kind == "purchase order":
        return [lookup, wire_call("match_purchase_order", invoice_id=invoice_id, purchase_order=case.requested_value)]
    if case.kind == "tax":
        return [lookup, wire_call("match_tax", invoice_id=invoice_id, expected_tax_minor=case.requested_value)]
    if case.kind == "currency":
        return [lookup, wire_call("match_currency", invoice_id=invoice_id, expected_currency=case.requested_value)]
    return []


def evidence_summary(case, actions, observations):
    """Make the rule teacher's terminal response from the observations."""
    if case.kind == "clarification":
        return wire_final("clarify")
    if case.kind == "no call":
        return wire_final("no_call")
    if not actions or not all(result["ok"] for result in observations):
        return wire_final("clarify")
    if case.kind == "lookup":
        return wire_final("found", observations[-1]["invoice_id"])
    outcome = observations[-1]
    return wire_final("match" if outcome["match"] else "mismatch", outcome["invoice_id"], outcome["value"])


def accepted_final(case):
    """Build the expected terminal record without executing the rollout."""
    if case.kind == "clarification":
        return wire_final("clarify")
    if case.kind == "no call":
        return wire_final("no_call")
    if case.kind == "lookup":
        return wire_final("found", case.target.invoice_id)
    actual = {
        "amount": case.target.total_minor, "purchase order": case.target.purchase_order,
        "tax": case.target.tax_minor, "currency": case.target.currency,
    }[case.kind]
    decision = "match" if actual is not None and actual == case.requested_value else "mismatch"
    return wire_final(decision, case.target.invoice_id, actual)


def score_episode(case, episode):
    expected = wanted_calls(case)
    tools_right = [a["call"] for a in episode.actions] == [a["call"] for a in expected]
    exact_calls = list(episode.actions) == expected
    valid_exec = (bool(episode.observations) or not expected) and all(
        observation["ok"] for observation in episode.observations
    )
    terminal = episode.final == accepted_final(case)
    safe_abstention = (not episode.actions and terminal) if not expected else None
    return {
        "structured": not episode.malformed and episode.final is not None,
        "tool_selection": tools_right,
        "arguments_exact": exact_calls,
        "execution_ok": valid_exec,
        "terminal_exact": terminal,
        "abstention_exact": safe_abstention,
        "success": not episode.malformed and exact_calls and valid_exec and terminal,
        "turns": len(episode.actions) + int(episode.final is not None),
    }


def teacher_episode(case, allowed, *, corrupt=False):
    planned = wanted_calls(case)
    if corrupt and planned:
        planned = [wire_call(a["call"], **{**a["args"], "invoice_id": case.decoy.invoice_id}) for a in planned]

    def choose(actions, observations):
        if len(actions) < len(planned):
            return planned[len(actions)]
        if corrupt and not planned:
            return wire_final("no_call" if case.kind == "clarification" else "clarify")
        return evidence_summary(case, actions, observations)

    return execute_episode(case, allowed, choose)


def rules_episode(case, allowed):
    """Parse the fixture's canonical utterances and derive a decision from the tool result."""
    patterns = (
        (r"Check whether the ledger contains (\S+)\.", "lookup_invoice", ()),
        (r"Check that (\S+) has a printed total of (\d+) minor units in ([A-Z]{3})\.", "match_amount",
         (("expected_minor", int), ("currency", str))),
        (r"Verify whether (\S+) belongs to (PO-[A-F0-9]+)\.", "match_purchase_order", (("purchase_order", str),)),
        (r"Check the printed tax on (\S+) against (\d+) minor units\.", "match_tax", (("expected_tax_minor", int),)),
        (r"Check whether (\S+) is denominated in ([A-Z]{3})\.", "match_currency", (("expected_currency", str),)),
    )
    plan = []
    for pattern, name, fields in patterns:
        match = re.match(pattern, case.request)
        if match is None or name not in allowed:
            continue
        invoice_id, *values = match.groups()
        plan.append(wire_call("lookup_invoice", invoice_id=invoice_id))
        if fields:
            args = {key: parse(value) for (key, parse), value in zip(fields, values, strict=True)}
            plan.append(wire_call(name, invoice_id=invoice_id, **args))
        break

    def choose(actions, observations):
        if not plan:
            return wire_final("no_call" if case.request.startswith("Explain in general") else "clarify")
        if observations and not observations[-1]["ok"]:
            return wire_final("clarify")
        if len(actions) < len(plan):
            return plan[len(actions)]
        last = observations[-1]
        if len(plan) == 1:
            return wire_final("found", last["invoice_id"])
        return wire_final("match" if last["match"] else "mismatch", last["invoice_id"], last["value"])

    return execute_episode(case, allowed, choose)

In [8]:
"""Encode assistant actions and evaluate a real read-only tool loop."""
from __future__ import annotations

TOOL_OUTPUT_LIMIT = 96
TOOL_SYSTEM = """
Check the invoice ledger only when the user has requested a lookup or a comparison and
provided a target invoice identifier. Mentioned alternate identifiers are different invoices.
A comparison requires a successful lookup for the target followed by a comparison tool.
Reply with one JSON object per turn. To call a tool use {"call":"tool_name","args":{...}}.
After lookup alone use {"final":"found","invoice_id":"the target identifier","value":null}.
After comparison use {"final":"match" or "mismatch","invoice_id":"the target identifier","value":the observed value}.
Read the actual comparison outcome and value from the tool response. Use
{"final":"clarify","invoice_id":null,"value":null} for a check with no invoice identifier.
Use {"final":"no_call","invoice_id":null,"value":null} for general questions. No prose or fences.
""".strip()


def tool_schema(allowed):
    wire_types = {str: "string", int: "integer"}
    schemas = []
    for name in allowed:
        fields, description = TOOL_CONTRACTS[name]
        schemas.append({
            "name": name, "description": description,
            "args": {key: wire_types[python_type] for key, python_type in fields.items()},
        })
    return schemas


def tool_messages(case, allowed, actions, observations, *, wrap_result=False):
    messages = [
        {"role": "system", "content": TOOL_SYSTEM + "\nAvailable read-only tools:\n" + canonical_json(tool_schema(allowed))},
        {"role": "user", "content": case.request},
    ]
    for action, observation in zip(actions, observations, strict=True):
        messages.append({"role": "assistant", "content": canonical_json(action)})
        messages.append({"role": "tool", "content": canonical_json({"result": observation} if wrap_result else observation)})
    return messages


def encode_episode(tokenizer, case, allowed, episode):
    turns = []
    for index, response in enumerate((*episode.actions, episode.final)):
        if response is None:
            raise ValueError(f"Trace {case.key} has no terminal response.")
        messages = tool_messages(case, allowed, episode.actions[:index], episode.observations[:index])
        prompt = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
        completion = tokenizer.encode(canonical_json(response), add_special_tokens=False) + [tokenizer.eos_token_id]
        if len(completion) > TOOL_OUTPUT_LIMIT or len(prompt) + len(completion) > MAX_TOKENS:
            raise ValueError(f"Trace {case.key} exceeds the fixed token budget.")
        turns.append(Turn(prompt, completion))
    sequences = split_turns(turns)
    if sum(sum(sequence.completion_mask) for sequence in sequences) != sum(len(turn.completion) for turn in turns):
        raise ValueError(f"Trace {case.key} lost supervised response tokens.")
    return sequences


def paired_tool_schedules(tokenizer, cases):
    examples = {"raw": [], "verified": []}
    audit = []
    for index, case in enumerate(cases):
        raw = teacher_episode(case, TRAIN_TOOLS, corrupt=index % 3 == 0)
        verified = teacher_episode(case, TRAIN_TOOLS)
        raw_score = score_episode(case, raw)
        verified_score = score_episode(case, verified)
        if not verified_score["success"] or raw_score["turns"] != verified_score["turns"]:
            raise ValueError(f"Training traces differ in required structure or verification failed for {case.key}.")
        if raw_score["success"] != (index % 3 != 0):
            raise ValueError(f"Training corruption did not match the audit for {case.key}.")
        examples["raw"].append(encode_episode(tokenizer, case, TRAIN_TOOLS, raw))
        examples["verified"].append(encode_episode(tokenizer, case, TRAIN_TOOLS, verified))
        audit.append({
            "case": case.key, "kind": case.kind, "raw": raw_score, "verified": verified_score,
        })
    schedules = {
        name: [[sequence for trace in update for sequence in trace] for update in make_schedule(traces)]
        for name, traces in examples.items()
    }
    return schedules, audit


def model_episode(model, tokenizer, case, allowed, *, wrap_result=False):
    device = model.get_input_embeddings().weight.device
    completions = []

    def choose(actions, observations):
        messages = tool_messages(case, allowed, actions, observations, wrap_result=wrap_result)
        prompt = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
        if len(prompt) + TOOL_OUTPUT_LIMIT > MAX_TOKENS:
            raise ValueError(f"Evaluation trace {case.key} exceeds the fixed token budget.")
        ids = torch.tensor([prompt], dtype=torch.long, device=device)
        with torch.inference_mode():
            generated = model.generate(
                input_ids=ids, attention_mask=torch.ones_like(ids), do_sample=False, num_beams=1,
                max_new_tokens=TOOL_OUTPUT_LIMIT, pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id, use_cache=True,
                temperature=None, top_p=None, top_k=None,
            )
        tokens = generated[0, len(prompt):].tolist()
        eos = tokenizer.eos_token_id
        if eos in tokens:
            tokens = tokens[:tokens.index(eos)]
        response = tokenizer.decode(tokens, skip_special_tokens=False).strip()
        completions.append(response)
        return response

    return execute_episode(case, allowed, choose), completions


def ledger_predictions(predict, cases):
    results = []
    for case in cases:
        allowed = (*TRAIN_TOOLS, *HELD_TOOLS)
        episode, completions = predict(case, allowed)
        results.append({
            "case": case.key, "split": case.split, "family": case.family, "kind": case.kind, "request": case.request,
            "actions": episode.actions, "observations": episode.observations, "final": episode.final,
            "completions": completions, "expected_calls": wanted_calls(case), "expected_final": accepted_final(case),
            **score_episode(case, episode),
        })
    return results


def ledger_summary(results):
    slices = {
        "seen_operations": [row for row in results if row["kind"] in {"lookup", "amount", "purchase order"}],
        "unseen_tax_operation": [row for row in results if row["kind"] == "tax"],
        "unseen_currency_operation": [row for row in results if row["kind"] == "currency"],
        "clarification": [row for row in results if row["kind"] == "clarification"],
        "no_call": [row for row in results if row["kind"] == "no call"],
    }
    metrics = ("structured", "tool_selection", "arguments_exact", "execution_ok", "terminal_exact", "success")
    return {
        name: {
            "n": len(rows),
            "count": {metric: sum(row[metric] for row in rows) for metric in metrics},
            "fraction": {metric: sum(row[metric] for row in rows) / len(rows) for metric in metrics},
        }
        for name, rows in slices.items() if rows
    }


def paired_tool_changes(raw, verified):
    def to_rows(rows):
        return [{"key": row["case"], "split": row["split"], "family": row["family"], "exact": row["success"]} for row in rows]

    by_kind = {}
    for kind in ("lookup", "amount", "purchase order", "tax", "currency", "clarification", "no call"):
        old = to_rows(row for row in raw if row["kind"] == kind)
        new = to_rows(row for row in verified if row["kind"] == kind)
        by_kind[kind] = {
            "n": len(old),
            "gain": (sum(row["exact"] for row in new) - sum(row["exact"] for row in old)) / len(old),
            "paired_family_interval": paired_family_interval(old, new),
        }
    return by_kind


def check_tool_rosters(train, held):
    seen_kinds = {"lookup", "amount", "purchase order", "clarification", "no call"}
    held_kinds = seen_kinds | {"tax", "currency"}
    for rows, expected in ((train, seen_kinds), (held, held_kinds)):
        if set(row.kind for row in rows) != expected or len({row.key for row in rows}) != len(rows):
            raise ValueError("Expected each question type and unique cases within each roster.")
    if ({row.key for row in train} & {row.key for row in held}
            or {row.family for row in train} & {row.family for row in held}):
        raise ValueError("Training and evaluation contain overlapping cases or families.")
    if set(HELD_TOOLS) & set(TRAIN_TOOLS):
        raise ValueError("Held-out tool schemas appear in the training tool pack.")
    return {"train": dict(Counter(row.kind for row in train)),
            "held": dict(Counter(row.kind for row in held))}


def tool_trace_audit_by_kind(audit):
    totals = defaultdict(lambda: {"count": 0, "raw_execution_ok": 0, "raw_success": 0, "verified_success": 0})
    for row in audit:
        bucket = totals[row["kind"]]
        bucket["count"] += 1
        bucket["raw_execution_ok"] += int(row["raw"]["execution_ok"])
        bucket["raw_success"] += int(row["raw"]["success"])
        bucket["verified_success"] += int(row["verified"]["success"])
    return dict(totals)

## Freeze the rosters

The fit rows use the training split. All three learned and prompted arms see identical held
families and schemas. Tax and currency checks appear only at evaluation. The original invoice extraction prompt
gives a regression check alongside the tool task.

In [9]:
train_cases = cases_for(examples_for("train", 90))
probe_items = [item for split in ("familiar", "novel", "independent") for item in examples_for(split, 18)]
held_cases = cases_for(probe_items, unseen_tool=True)
regression_items = [item for split in ("familiar", "novel", "independent") for item in examples_for(split, 4)]
coverage = check_tool_rosters(train_cases, held_cases)
print("Fit by question", coverage["train"])
print("Evaluation by question", coverage["held"])
print("Evaluation by slice and question", dict(Counter((case.split, case.kind) for case in held_cases)))
opening_case = next(case for case in held_cases if case.key == "SF-429916")
opening_replays = {
    "request": opening_case.request,
    "expected": asdict(teacher_episode(opening_case, TRAIN_TOOLS)),
    "decoy": asdict(teacher_episode(opening_case, TRAIN_TOOLS, corrupt=True)),
}
print(json.dumps(opening_replays, indent=2))

Fit by question {'lookup': 18, 'amount': 18, 'purchase order': 18, 'clarification': 18, 'no call': 18}
Evaluation by question {'lookup': 6, 'amount': 6, 'purchase order': 6, 'clarification': 6, 'no call': 6, 'tax': 12, 'currency': 12}
Evaluation by slice and question {('familiar', 'lookup'): 2, ('familiar', 'amount'): 2, ('familiar', 'purchase order'): 2, ('familiar', 'clarification'): 2, ('familiar', 'no call'): 2, ('familiar', 'tax'): 4, ('familiar', 'currency'): 4, ('novel', 'lookup'): 2, ('novel', 'amount'): 2, ('novel', 'purchase order'): 2, ('novel', 'clarification'): 2, ('novel', 'no call'): 2, ('novel', 'tax'): 4, ('novel', 'currency'): 4, ('independent', 'lookup'): 2, ('independent', 'amount'): 2, ('independent', 'purchase order'): 2, ('independent', 'clarification'): 2, ('independent', 'no call'): 2, ('independent', 'tax'): 4, ('independent', 'currency'): 4}
{
  "request": "Check that SF-429916 has a printed total of 234976 minor units in GBP. The notes also mention SF-862527

## Train on raw and verified traces

For a fixed third of the training roster, replace the correct record with the decoy, or swap
the two no-call outcomes. The decoy can return a real value, so successful execution does
not prove task correctness. Replay collects observations in both arms. The verified arm must
also pass exact action, argument and terminal checks. The grouped optimizer batches preserve
the same cases and assistant-turn count. Report actual token counts and successful updates.

In [10]:
schedules, training_audit = paired_tool_schedules(tokenizer, train_cases)
teacher_audit_by_kind = tool_trace_audit_by_kind(training_audit)
print("Pre-fit teacher versus independent scorer", json.dumps(teacher_audit_by_kind, indent=2))
print("Updates per arm", {name: len(schedule) for name, schedule in schedules.items()})
example = next(case for case in train_cases if case.kind == "amount")
replay = teacher_episode(example, TRAIN_TOOLS, corrupt=True)
print(json.dumps({
    "request": example.request, "actions": replay.actions, "observations": replay.observations,
    "final": replay.final, "score": score_episode(example, replay),
}, indent=2))

Pre-fit teacher versus independent scorer {
  "lookup": {
    "count": 18,
    "raw_execution_ok": 18,
    "raw_success": 12,
    "verified_success": 18
  },
  "amount": {
    "count": 18,
    "raw_execution_ok": 18,
    "raw_success": 12,
    "verified_success": 18
  },
  "purchase order": {
    "count": 18,
    "raw_execution_ok": 18,
    "raw_success": 12,
    "verified_success": 18
  },
  "clarification": {
    "count": 18,
    "raw_execution_ok": 18,
    "raw_success": 12,
    "verified_success": 18
  },
  "no call": {
    "count": 18,
    "raw_execution_ok": 18,
    "raw_success": 12,
    "verified_success": 18
  }
}
Updates per arm {'raw': 45, 'verified': 45}
{
  "request": "Check that SF-280550 has a printed total of 79405 minor units in GBP. The notes also mention SF-327852. That is a different invoice.",
  "actions": [
    {
      "call": "lookup_invoice",
      "args": {
        "invoice_id": "SF-327852"
      }
    },
    {
      "call": "match_amount",
      "args": {
    

## Check one update on the longest training batch

Use the widest padded training batch from either branch. Record the synchronised step and
the allocator peaks. Free the profiling model before loading the models being compared.

In [11]:
output = Path("results")
output.mkdir(exist_ok=True)
free, _ = torch.cuda.mem_get_info()
if free < 10 * 2**30:
    raise RuntimeError("The tool-call update profile needs at least 10 GiB free on T4.")
profile_arm, batch = max(
    ((name, batch) for name, schedule in schedules.items() for batch in schedule),
    key=lambda item: max(len(sequence.input_ids) for sequence in item[1]) * len(item[1]),
)
probe = load_model("fp16", trainable=True)
try:
    probe_metrics = adapter_train(
        probe, [batch], sft_objective(tokenizer.pad_token_id, "cuda"),
        learning_rate=2e-4, use_cuda=True,
    )
    profile = {
        "arm": profile_arm, "sequences": len(batch),
        "max_sequence_tokens": max(len(sequence.input_ids) for sequence in batch),
        "peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
        **probe_metrics,
    }
finally:
    del probe
    free_gpu()
(output / "14-t4-profile.json").write_text(json.dumps(profile, indent=2), encoding="utf-8")
print("T4 profile", json.dumps(profile, indent=2))
if (profile["updates"] != 1 or profile["seconds"] > 45 or profile["peak_gib"] > 9
        or profile["peak_reserved_gib"] > min(11, free / 2**30 - 1)):
    raise RuntimeError("The update profile exceeded the registered T4 budget.")

T4 profile {
  "arm": "raw",
  "sequences": 2,
  "max_sequence_tokens": 515,
  "peak_reserved_gib": 4.17578125,
  "attempted_updates": 1,
  "updates": 1,
  "supervised_response_tokens": 174,
  "unpadded_input_tokens": 1022,
  "seconds": 2.187135029999922,
  "supervised_tokens_per_second": 79.55613056044656,
  "peak_gib": 3.839597702026367,
  "losses": [
    0.2790513336658478
  ],
  "trainable_parameters": 540672
}


## Run all three arms on the same cases

Local rules parse the canonical request wording, including explicitly programmed tax
and currency comparators. The rules score provides a fixture diagnostic. The prompted arm starts from
the pinned instruct checkpoint. The two LoRA arms begin from
the same checkpoint and use the same configuration. Every evaluation turn generates a fresh
assistant decision. The runtime executes parsed calls and adds the actual response to the
next input.

In [12]:
results = {"local_rules": ledger_predictions(lambda case, tools: (rules_episode(case, tools), []), held_cases)}
extraction, training, harness_diagnostic = {}, {}, {}
print("Canonical rules", json.dumps(ledger_summary(results["local_rules"]), indent=2))

def evaluate_arm(name, model):
    model.eval()
    results[name] = ledger_predictions(lambda case, tools: model_episode(model, tokenizer, case, tools), held_cases)
    extraction[name] = predictions(model, tokenizer, regression_items, exemplar)
    print(name, json.dumps(ledger_summary(results[name]), indent=2))

Canonical rules {
  "seen_operations": {
    "n": 18,
    "count": {
      "structured": 18,
      "tool_selection": 18,
      "arguments_exact": 18,
      "execution_ok": 18,
      "terminal_exact": 18,
      "success": 18
    },
    "fraction": {
      "structured": 1.0,
      "tool_selection": 1.0,
      "arguments_exact": 1.0,
      "execution_ok": 1.0,
      "terminal_exact": 1.0,
      "success": 1.0
    }
  },
  "unseen_tax_operation": {
    "n": 12,
    "count": {
      "structured": 12,
      "tool_selection": 12,
      "arguments_exact": 12,
      "execution_ok": 12,
      "terminal_exact": 12,
      "success": 12
    },
    "fraction": {
      "structured": 1.0,
      "tool_selection": 1.0,
      "arguments_exact": 1.0,
      "execution_ok": 1.0,
      "terminal_exact": 1.0,
      "success": 1.0
    }
  },
  "unseen_currency_operation": {
    "n": 12,
    "count": {
      "structured": 12,
      "tool_selection": 12,
      "arguments_exact": 12,
      "execution_ok": 12,
   

In [13]:
model = load_model("fp16")
try:
    evaluate_arm("prompted", model)
finally:
    del model
    free_gpu()

prompted {
  "seen_operations": {
    "n": 18,
    "count": {
      "structured": 18,
      "tool_selection": 0,
      "arguments_exact": 0,
      "execution_ok": 0,
      "terminal_exact": 4,
      "success": 0
    },
    "fraction": {
      "structured": 1.0,
      "tool_selection": 0.0,
      "arguments_exact": 0.0,
      "execution_ok": 0.0,
      "terminal_exact": 0.2222222222222222,
      "success": 0.0
    }
  },
  "unseen_tax_operation": {
    "n": 12,
    "count": {
      "structured": 12,
      "tool_selection": 0,
      "arguments_exact": 0,
      "execution_ok": 0,
      "terminal_exact": 0,
      "success": 0
    },
    "fraction": {
      "structured": 1.0,
      "tool_selection": 0.0,
      "arguments_exact": 0.0,
      "execution_ok": 0.0,
      "terminal_exact": 0.0,
      "success": 0.0
    }
  },
  "unseen_currency_operation": {
    "n": 12,
    "count": {
      "structured": 12,
      "tool_selection": 0,
      "arguments_exact": 0,
      "execution_ok": 0,
      "t

In [14]:
for name in ("raw", "verified"):
    model = load_model("fp16", trainable=True)
    try:
        training[name] = adapter_train(
            model, schedules[name], sft_objective(tokenizer.pad_token_id, "cuda"),
            learning_rate=2e-4, use_cuda=True,
        )
        evaluate_arm(name, model)
        if name == "verified":
            wrapped = ledger_predictions(
                lambda case, tools: model_episode(model, tokenizer, case, tools, wrap_result=True), held_cases
            )
            canonical = results[name]
            pairs = list(zip(canonical, wrapped, strict=True))
            assert all(before["case"] == after["case"] for before, after in pairs)
            harness_diagnostic = {
                "gained_case_ids": [after["case"] for before, after in pairs if after["success"] and not before["success"]],
                "lost_case_ids": [after["case"] for before, after in pairs if before["success"] and not after["success"]],
                "baseline_contract": "ledger-v1", "mutated_contract": "ledger-v2-result-envelope",
                "mutation": "Wrap tool observations in a result object before the model sees them.",
                "canonical_pass_at_1": ledger_summary(results[name]),
                "wrapped_pass_at_1": ledger_summary(wrapped),
                "paired_changes": paired_tool_changes(results[name], wrapped),
                "wrapped_episodes": wrapped,
            }
            print("Harness result envelope", json.dumps({
                key: value for key, value in harness_diagnostic.items() if key != "wrapped_episodes"
            }, indent=2))
    finally:
        del model
        free_gpu()
for name, info in training.items():
    if info["updates"] != len(schedules[name]):
        raise RuntimeError(f"Unexpected skipped update in the {name} branch.")
print("Fit details", {name: {key: value for key, value in info.items() if key != "losses"}
                      for name, info in training.items()})

raw {
  "seen_operations": {
    "n": 18,
    "count": {
      "structured": 18,
      "tool_selection": 13,
      "arguments_exact": 1,
      "execution_ok": 18,
      "terminal_exact": 1,
      "success": 1
    },
    "fraction": {
      "structured": 1.0,
      "tool_selection": 0.7222222222222222,
      "arguments_exact": 0.05555555555555555,
      "execution_ok": 1.0,
      "terminal_exact": 0.05555555555555555,
      "success": 0.05555555555555555
    }
  },
  "unseen_tax_operation": {
    "n": 12,
    "count": {
      "structured": 12,
      "tool_selection": 0,
      "arguments_exact": 0,
      "execution_ok": 0,
      "terminal_exact": 0,
      "success": 0
    },
    "fraction": {
      "structured": 1.0,
      "tool_selection": 0.0,
      "arguments_exact": 0.0,
      "execution_ok": 0.0,
      "terminal_exact": 0.0,
      "success": 0.0
    }
  },
  "unseen_currency_operation": {
    "n": 12,
    "count": {
      "structured": 8,
      "tool_selection": 0,
      "arguments_

verified {
  "seen_operations": {
    "n": 18,
    "count": {
      "structured": 18,
      "tool_selection": 14,
      "arguments_exact": 14,
      "execution_ok": 14,
      "terminal_exact": 14,
      "success": 14
    },
    "fraction": {
      "structured": 1.0,
      "tool_selection": 0.7777777777777778,
      "arguments_exact": 0.7777777777777778,
      "execution_ok": 0.7777777777777778,
      "terminal_exact": 0.7777777777777778,
      "success": 0.7777777777777778
    }
  },
  "unseen_tax_operation": {
    "n": 12,
    "count": {
      "structured": 10,
      "tool_selection": 0,
      "arguments_exact": 0,
      "execution_ok": 0,
      "terminal_exact": 0,
      "success": 0
    },
    "fraction": {
      "structured": 0.8333333333333334,
      "tool_selection": 0.0,
      "arguments_exact": 0.0,
      "execution_ok": 0.0,
      "terminal_exact": 0.0,
      "success": 0.0
    }
  },
  "unseen_currency_operation": {
    "n": 12,
    "count": {
      "structured": 12,
      "t

Harness result envelope {
  "gained_case_ids": [],
  "lost_case_ids": [],
  "baseline_contract": "ledger-v1",
  "mutated_contract": "ledger-v2-result-envelope",
  "mutation": "Wrap tool observations in a result object before the model sees them.",
  "canonical_pass_at_1": {
    "seen_operations": {
      "n": 18,
      "count": {
        "structured": 18,
        "tool_selection": 14,
        "arguments_exact": 14,
        "execution_ok": 14,
        "terminal_exact": 14,
        "success": 14
      },
      "fraction": {
        "structured": 1.0,
        "tool_selection": 0.7777777777777778,
        "arguments_exact": 0.7777777777777778,
        "execution_ok": 0.7777777777777778,
        "terminal_exact": 0.7777777777777778,
        "success": 0.7777777777777778
      }
    },
    "unseen_tax_operation": {
      "n": 12,
      "count": {
        "structured": 10,
        "tool_selection": 0,
        "arguments_exact": 0,
        "execution_ok": 0,
        "terminal_exact": 0,
      

Fit details {'raw': {'attempted_updates': 45, 'updates': 45, 'supervised_response_tokens': 4441, 'unpadded_input_tokens': 37319, 'seconds': 9.00235211100005, 'supervised_tokens_per_second': 493.31551857136367, 'peak_gib': 3.8477578163146973, 'trainable_parameters': 540672}, 'verified': {'attempted_updates': 45, 'updates': 45, 'supervised_response_tokens': 4440, 'unpadded_input_tokens': 37320, 'seconds': 9.14779253200004, 'supervised_tokens_per_second': 485.36299708026445, 'peak_gib': 3.8477578163146973, 'trainable_parameters': 540672}}


## Read the measurements

Exact trajectory success requires correct ordered calls, valid execution and the correct
evidence-linked final record. Inspect arguments when execution exceeds end-to-end success.
Compare held-out tax and currency separately from the trained operations. Check that new behavior does
not come with worse extraction on the same original task. These synthetic counts are small,
especially the no-call rails, and estimates are descriptive.

In [15]:
import matplotlib.pyplot as plt

summaries = {name: ledger_summary(rows) for name, rows in results.items()}
changes = paired_tool_changes(results["raw"], results["verified"])
print("Verified minus raw trajectory success by question", json.dumps(changes, indent=2))
receipt = {
    "model": MODEL, "revision": REVISION, "seed": SEED, "gpu": torch.cuda.get_device_name(0),
    "versions": {package: importlib.metadata.version(package) for package in ("torch", "transformers", "peft", "accelerate")},
    "tool_turn_limit": TURN_LIMIT, "generated_tokens_per_turn": TOOL_OUTPUT_LIMIT,
    "train_tools": TRAIN_TOOLS, "held_tools": HELD_TOOLS, "training_case_audit": training_audit,
    "teacher_audit_by_kind": teacher_audit_by_kind, "coverage_by_kind": coverage,
    "t4_profile": profile, "opening_replays": opening_replays,
    "training": training, "metrics": summaries, "paired_raw_to_verified": changes,
    "episodes": results, "extraction_regression": extraction,
    "harness_version_diagnostic": harness_diagnostic,
}
(output / "14-tool-call-results.json").write_text(json.dumps(receipt, indent=2), encoding="utf-8")
print("Extraction exact by arm", {
    name: (sum(row["exact"] for row in rows), len(rows)) for name, rows in extraction.items()
})

Verified minus raw trajectory success by question {
  "lookup": {
    "n": 6,
    "gain": 0.8333333333333334,
    "paired_family_interval": [
      0.5,
      1.0
    ]
  },
  "amount": {
    "n": 6,
    "gain": 1.0,
    "paired_family_interval": [
      1.0,
      1.0
    ]
  },
  "purchase order": {
    "n": 6,
    "gain": 0.3333333333333333,
    "paired_family_interval": [
      0.0,
      0.6666666666666666
    ]
  },
  "tax": {
    "n": 12,
    "gain": 0.0,
    "paired_family_interval": [
      0.0,
      0.0
    ]
  },
  "currency": {
    "n": 12,
    "gain": 1.0,
    "paired_family_interval": [
      1.0,
      1.0
    ]
  },
  "clarification": {
    "n": 6,
    "gain": 0.0,
    "paired_family_interval": [
      0.0,
      0.0
    ]
  },
  "no call": {
    "n": 6,
    "gain": 0.0,
    "paired_family_interval": [
      0.0,
      0.0
    ]
  }
}
Extraction exact by arm {'prompted': (0, 12), 'raw': (0, 12), 'verified': (0, 12)}


In [16]:
fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
colors = {"prompted": "#486782", "raw": "#C68143", "verified": "#156B62"}
slices = list(summaries["prompted"])
for offset, name in enumerate(colors):
    summary = summaries[name]
    positions = [index + 0.23 * (offset - 1) for index in range(len(slices))]
    ax.bar(positions, [summary[k]["fraction"]["success"] for k in slices], width=0.21, label=name, color=colors[name])
ax.set_title("Exact trajectory success by decision surface", loc="left")
ax.set_ylim(0, 1.03)
ax.set_ylabel("Fraction of cases")
labels = ["Seen tools", "Unseen tax", "Unseen currency", "Clarification", "No call"]
ax.set_xticks(range(len(slices)), [
    f"{label}\n(n={summaries['prompted'][key]['n']})" for label, key in zip(labels, slices, strict=True)
])
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1), frameon=False)
fig.savefig(output / "trajectory-success.svg")
plt.show()

sample = next((row for row in results["verified"] if not row["success"]), None)
if sample:
    print(json.dumps(sample, indent=2))

<Figure size 1000x450 with 1 Axes>

{
  "case": "SF-862527",
  "split": "familiar",
  "family": "familiar-02",
  "kind": "purchase order",
  "request": "Verify whether SF-862527 belongs to PO-B3AD1B171. The notes also mention SF-112096. That is a different invoice.",
  "actions": [
    {
      "call": "lookup_invoice",
      "args": {
        "invoice_id": "SF-862527"
      }
    },
    {
      "call": "match_invoice",
      "args": {
        "invoice_id": "SF-862527",
        "target_identifier": "PO-B3AD1B171"
      }
    }
  ],
  "observations": [
    {
      "ok": true,
      "invoice_id": "SF-862527",
      "retrieved": true
    },
    {
      "ok": false,
      "error": "unknown_tool"
    }
  ],
  "final": {
    "final": "no_call",
    "invoice_id": null,
    "value": null
  },
  "completions": [
    "{\"call\":\"lookup_invoice\",\"args\":{\"invoice_id\":\"SF-862527\"}}",
    "{\"call\":\"match_invoice\",\"args\":{\"invoice_id\":\"SF-862527\",\"target_identifier\":\"PO-B3AD1B171\"}}",
    "{\"final\":\"no_call\",\"

## Read the result

Review one replay where the tool executed and the request still failed. Identify the first
wrong argument, then follow its observed value into the final response.

[Read the lesson](https://docs.recohut.com/courses/advanced-llm-fine-tuning/3-adapt-models/14-train-on-verified-tool-calls)

## Sources

The multi-turn masking helper adapts code from the Apache-2.0
[Together Post-Training Cookbook](https://github.com/togethercomputer/post-training-cookbook/tree/226e03ca225d6f13434047cf0f45ab3070a29801).
All invoice records, tool schemas, corruptions and evaluation cases in this lesson are synthetic.
The experiment uses the pinned [Qwen2.5-0.5B-Instruct weights](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/tree/7ae557604adf67be50417f59c2c2f167def9a775).